# Phi-4 Multimodal × POPE: phân tích lỗi → phát hiện lỗi → cải thiện

Notebook v2, sắp xếp lại theo đúng 3 yêu cầu của thầy:

| Bước | Yêu cầu của thầy | Phần trong notebook |
|---|---|---|
| 1 | Chạy benchmark (baseline) | Phần 1 (giữ nguyên code cũ) |
| 2 | Model đang lỗi gì, lỗi chỗ nào, **list rõ từng case** | Phần A |
| 3 | Dùng **kỹ thuật gì để phát hiện** các lỗi đó | Phần B |
| 4 | Từ các lỗi thấy được, **cải thiện model** thế nào | Phần C |

**Thay đổi so với notebook cũ**
- Bỏ phần pilot 60 mẫu, vì 900 mẫu đã bao phủ.
- Bỏ các cell dedup/annotation rời rạc, gom vào Phần A.
- Sửa lỗi phân tích theo category. POPE dùng chung câu hỏi *positive* cho cả 3 split, chỉ *negative* khác nhau. Vì vậy phải đánh giá trên cặp (ảnh, câu hỏi) duy nhất, và so sánh FP theo category trên các hàng negative của 900 mẫu.
- Mỗi lỗi được gán lý do tự động (có số liệu) và xuất báo cáo HTML có ảnh, khung bbox của vật thể bị bỏ sót.

**Cần GPU** (Kaggle T4 x2) và bật Internet (tải COCO annotations, DETR).

In [ ]:

!nvidia-smi


In [ ]:

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    gpu = torch.cuda.get_device_properties(i)
    print(f"\nGPU {i}: {gpu.name}")
    print(f"VRAM: {gpu.total_memory / 1024**3:.2f} GB")


In [ ]:

!pip install -q \
    transformers==4.48.2 \
    accelerate==1.3.0 \
    pillow==11.1.0 \
    soundfile==0.13.1 \
    scipy==1.15.2 \
    peft==0.13.2 \
    backoff==2.2.1 \
    einops


In [ ]:
import torch
import transformers
import accelerate
import PIL
import huggingface_hub

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("Pillow:", PIL.__version__)
print("HuggingFace Hub:", huggingface_hub.__version__)

print("\nCUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}: {torch.cuda.get_device_name(i)} - "
        f"{torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB"
    )

In [ ]:
from huggingface_hub import model_info

MODEL_ID = "microsoft/Phi-4-multimodal-instruct"

try:
    info = model_info(MODEL_ID)
    print("Model:", info.id)
    print("Revision:", info.sha)
    print("Truy cập Hugging Face: OK")
except Exception as e:
    print("Lỗi truy cập:", repr(e))

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForCausalLM

MODEL_ID = "microsoft/Phi-4-multimodal-instruct"
MODEL_REVISION = "93f923e1a7727d1c4f446756212d9d3e8fcc5d81"

print("Đang tải Processor...")

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=True
)

print("Processor tải thành công!")
print("Đang tải Phi-4 Model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory={
        0: "10GiB",
        1: "10GiB",
        "cpu": "30GiB"
    },
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    _attn_implementation="eager"
)

model.eval()

print("\n===== MODEL LOADED SUCCESSFULLY =====")
print("Model:", MODEL_ID)
print("Dtype:", next(model.parameters()).dtype)
print("Device map:", model.hf_device_map)

print("\n===== GPU MEMORY =====")
for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}: "
        f"{torch.cuda.memory_allocated(i) / 1024**3:.2f} GB allocated / "
        f"{torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB"
    )

## Phần 1. Bước 1: Benchmark baseline (code cũ, giữ nguyên)
Chạy 300 mẫu × 3 split (random / popular / adversarial) = 900 mẫu. Kết quả ghi vào `predictions_900.jsonl`, chạy lại sẽ resume.

In [ ]:

import os
import re
import json
import time
import torch
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm
from datasets import load_dataset
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# =====================================================
# 1. CONFIGURATION
# =====================================================

MODEL_ID = "microsoft/Phi-4-multimodal-instruct"
MODEL_REVISION = "93f923e1a7727d1c4f446756212d9d3e8fcc5d81"

DATASET_ID = "lmms-lab-encoder/POPE"
DATASET_REVISION = "4db1276"

CATEGORIES = ["random", "popular", "adversarial"]

SAMPLES_PER_CATEGORY = 300
RANDOM_SEED = 42
EXPECTED_TOTAL = SAMPLES_PER_CATEGORY * len(CATEGORIES)

# Thư mục riêng cho benchmark 900 mẫu
OUTPUT_DIR = Path("/kaggle/working/phi4_pope_900")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_FILE = OUTPUT_DIR / "predictions_900.jsonl"
METRICS_FILE = OUTPUT_DIR / "metrics_900.json"
CONFIG_FILE = OUTPUT_DIR / "run_config_900.json"

# =====================================================
# 2. CHECK MODEL
# =====================================================

assert model is not None, "Model chưa được load!"
assert processor is not None, "Processor chưa được load!"
assert torch.cuda.is_available(), "Không tìm thấy GPU!"

model.eval()

print("=" * 55)
print("PHI-4 MULTIMODAL - POPE BENCHMARK")
print("=" * 55)

print("Model:", MODEL_ID)
print("GPU:", torch.cuda.get_device_name(0))
print("Samples per category:", SAMPLES_PER_CATEGORY)
print("Expected total:", EXPECTED_TOTAL)

# =====================================================
# 3. LOAD DATASET
# =====================================================

print("\nLoading POPE dataset...")

dataset = load_dataset(
    DATASET_ID,
    name="Full",
    revision=DATASET_REVISION
)

# Chọn cố định 300 mẫu cho mỗi category.
# Seed giống nhau => kết quả chọn mẫu có thể tái lập.

selected_dataset = {}

for category in CATEGORIES:

    category_data = dataset[category]

    assert len(category_data) >= SAMPLES_PER_CATEGORY, (
        f"{category} không đủ dữ liệu!"
    )

    selected = (
        category_data
        .shuffle(seed=RANDOM_SEED)
        .select(range(SAMPLES_PER_CATEGORY))
    )

    selected_dataset[category] = selected

    print(
        f"{category}: {len(selected)} / "
        f"{len(category_data)} samples selected"
    )

print("\nTotal selected:", sum(
    len(selected_dataset[c]) for c in CATEGORIES
))

# =====================================================
# 4. NORMALIZE ANSWER
# =====================================================

def normalize_answer(text):

    text = str(text).strip().lower()

    match = re.search(r"\b(yes|no)\b", text)

    if match:
        return match.group(1)

    return "unknown"

# =====================================================
# 5. INFERENCE FUNCTION
# =====================================================

def phi4_pope_inference(image, question):

    prompt = (
        "<|user|><|image_1|>"
        f"{question} "
        "Answer with only Yes or No."
        "<|end|><|assistant|>"
    )

    inputs = processor(
        text=prompt,
        images=[image.convert("RGB")],
        return_tensors="pt"
    ).to("cuda:0")

    with torch.inference_mode():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            use_cache=True
        )

    output_ids = output_ids[:, inputs["input_ids"].shape[1]:]

    raw_output = processor.batch_decode(
        output_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0].strip()

    prediction = normalize_answer(raw_output)

    return raw_output, prediction

# =====================================================
# 6. LOAD CHECKPOINT
# =====================================================

results = {}

if PREDICTIONS_FILE.exists():

    print("\nLoading existing checkpoint...")

    with open(
        PREDICTIONS_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            if not line.strip():
                continue

            record = json.loads(line)

            results[record["sample_key"]] = record

    print("Loaded latest records:", len(results))

# Chỉ bỏ qua các mẫu đã xử lý xong.
# Mẫu error sẽ được chạy lại.

completed = {
    key
    for key, record in results.items()
    if record["status"] in ["success", "unknown"]
}

print("Already completed:", len(completed))

# =====================================================
# 7. RUN BENCHMARK
# =====================================================

start_time = time.time()

total_processed = 0
total_errors = 0

with open(
    PREDICTIONS_FILE,
    "a",
    encoding="utf-8"
) as output_file:

    for category in CATEGORIES:

        category_data = selected_dataset[category]

        print(f"\n===== CATEGORY: {category.upper()} =====")

        for idx, sample in enumerate(
            tqdm(category_data, desc=category)
        ):

            sample_id = sample.get("id")
            question_id = sample.get("question_id")

            identifier = (
                question_id if question_id is not None
                else sample_id if sample_id is not None
                else idx
            )

            sample_key = f"{category}:{identifier}"

            # Bỏ qua mẫu đã hoàn thành
            if sample_key in completed:
                continue

            record = {
                "sample_key": sample_key,
                "id": sample_id,
                "question_id": question_id,
                "category": category,
                "question": sample["question"],
                "ground_truth": str(sample["answer"]).strip().lower(),
                "image_source": sample.get("image_source"),
                "raw_output": None,
                "prediction": "unknown",
                "status": "error",
                "error": None
            }

            try:

                raw_output, prediction = phi4_pope_inference(
                    sample["image"],
                    sample["question"]
                )

                record["raw_output"] = raw_output
                record["prediction"] = prediction

                record["status"] = (
                    "success"
                    if prediction in ["yes", "no"]
                    else "unknown"
                )

                record["error"] = None

            except Exception as e:

                record["status"] = "error"
                record["error"] = str(e)

                total_errors += 1

                if "out of memory" in str(e).lower():
                    torch.cuda.empty_cache()

            # Cập nhật kết quả mới nhất trong bộ nhớ
            results[sample_key] = record

            # Lưu ngay sau từng mẫu
            output_file.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

            output_file.flush()

            total_processed += 1

            if record["status"] in ["success", "unknown"]:
                completed.add(sample_key)

            if total_processed % 25 == 0:

                print(
                    f"\nProcessed this run: {total_processed} | "
                    f"Errors this run: {total_errors} | "
                    f"Completed total: {len(completed)}/{EXPECTED_TOTAL}"
                )

elapsed_time = time.time() - start_time

print("\n===== INFERENCE FINISHED =====")

print("Processed this run:", total_processed)
print("Errors this run:", total_errors)
print("Completed total:", len(completed))
print("Expected total:", EXPECTED_TOTAL)
print("Elapsed minutes:", round(elapsed_time / 60, 2))
print("Checkpoint:", PREDICTIONS_FILE)

# =====================================================
# 8. CALCULATE METRICS
# =====================================================

def calculate_metrics(records):

    df = pd.DataFrame(records)

    if len(df) == 0:
        return {
            "total": 0,
            "valid": 0
        }

    valid = df[
        df["status"].eq("success")
        & df["ground_truth"].isin(["yes", "no"])
        & df["prediction"].isin(["yes", "no"])
    ].copy()

    if len(valid) == 0:

        return {
            "total": len(df),
            "valid": 0,
            "unknown": int(
                (df["status"] == "unknown").sum()
            ),
            "errors": int(
                (df["status"] == "error").sum()
            )
        }

    y_true = valid["ground_truth"]
    y_pred = valid["prediction"]

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=["yes", "no"]
    )

    tp = int(cm[0, 0])
    fn = int(cm[0, 1])
    fp = int(cm[1, 0])
    tn = int(cm[1, 1])

    return {
        "total": len(df),
        "valid": len(valid),
        "unknown": int(
            (df["status"] == "unknown").sum()
        ),
        "errors": int(
            (df["status"] == "error").sum()
        ),
        "accuracy": float(
            accuracy_score(y_true, y_pred)
        ),
        "precision": float(
            precision_score(
                y_true,
                y_pred,
                pos_label="yes",
                zero_division=0
            )
        ),
        "recall": float(
            recall_score(
                y_true,
                y_pred,
                pos_label="yes",
                zero_division=0
            )
        ),
        "f1": float(
            f1_score(
                y_true,
                y_pred,
                pos_label="yes",
                zero_division=0
            )
        ),
        "yes_ratio": float(
            (y_pred == "yes").mean()
        ),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn
    }

# Chỉ sử dụng kết quả mới nhất theo sample_key.
all_records = list(results.values())

metrics = {}

for category in CATEGORIES:

    category_records = [
        r for r in all_records
        if r["category"] == category
    ]

    metrics[category] = calculate_metrics(
        category_records
    )

metrics["overall"] = calculate_metrics(all_records)

# Lưu metrics
with open(
    METRICS_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metrics,
        f,
        ensure_ascii=False,
        indent=4
    )

# =====================================================
# 9. SAVE RUN CONFIG
# =====================================================

run_config = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "dataset_id": DATASET_ID,
    "dataset_revision": DATASET_REVISION,
    "dataset_config": "Full",
    "categories": CATEGORIES,
    "samples_per_category": SAMPLES_PER_CATEGORY,
    "expected_total": EXPECTED_TOTAL,
    "random_seed": RANDOM_SEED,
    "dtype": "float16",
    "device_map": "auto",
    "max_new_tokens": 8,
    "do_sample": False,
    "prompt": "Answer with only Yes or No.",
    "checkpoint": "Append-only JSONL; resume skips success/unknown",
    "elapsed_seconds_this_run": elapsed_time,
    "completed_unique_samples": len(completed)
}

with open(
    CONFIG_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        run_config,
        f,
        ensure_ascii=False,
        indent=4
    )

# =====================================================
# 10. DISPLAY RESULTS
# =====================================================

print("\n===== FINAL METRICS =====")

metrics_df = pd.DataFrame(metrics).T

display(metrics_df)

print("\n===== STATUS =====")

status_df = pd.DataFrame(all_records)

if not status_df.empty:
    display(status_df["status"].value_counts())

print("\n===== CATEGORY COUNTS =====")

if not status_df.empty:
    display(
        status_df.groupby(
            ["category", "status"]
        ).size().unstack(fill_value=0)
    )

print("\n===== SAVED FILES =====")

for file in OUTPUT_DIR.iterdir():
    print(file.name, "-", file.stat().st_size, "bytes")

print("\nBenchmark 900 completed records:",
      len(completed), "/", EXPECTED_TOTAL)


## Phần A. Bước 2: Model đang lỗi gì, lỗi ở đâu, list từng case

Ba câu hỏi cần trả lời:
1. **Lỗi loại gì?** FN (có vật thể nhưng model nói *No*) hay FP (không có nhưng model nói *Yes*, tức hallucination).
2. **Lỗi ở đâu / vật thể nào?** Thống kê theo object, và vẽ bbox lên ảnh.
3. **Vì sao lỗi?** Gán lý do bằng số liệu từ COCO annotations (kích thước vật thể, co-occurrence), rồi so với nhóm trả lời đúng để biết lý do đó có thật sự phân biệt được lỗi hay không.

Lưu ý: lý do tự động chỉ là **giả thuyết**, cần xác nhận bằng mắt qua `error_report.html` (cột `manual_reason` để trống cho bước đó).

In [ ]:
# ===== A1. Load predictions, dedup đúng cách, baseline =====
import os, re, json, math, time, base64, io, zipfile, urllib.request
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd
from PIL import Image, ImageDraw
from IPython.display import display, HTML

BASE = Path("/kaggle/working/phi4_pope_900")
PRED_FILE = BASE / "predictions_900.jsonl"
OUT2 = BASE / "step2_error_listing"
OUT3 = BASE / "step3_detection"
OUT4 = BASE / "step4_improvement"
for d in (OUT2, OUT3, OUT4):
    d.mkdir(parents=True, exist_ok=True)
CATEGORIES = ["random", "popular", "adversarial"]

def norm_q(q):
    q = str(q).lower().strip().replace("imange", "image")   # 4 câu hỏi gốc bị typo "imange"
    q = re.sub(r"[?.!,;:]+$", "", q)
    return re.sub(r"\s+", " ", q)

def extract_obj(nq):
    m = re.search(r"(?:is|are) there (?:a|an|any) (.+?) in the image$", nq)
    return m.group(1).strip() if m else None

def metrics_w(y_true, y_pred, w=None):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    w = np.ones(len(y_true)) if w is None else np.asarray(w, float)
    tp = w[(y_true == "yes") & (y_pred == "yes")].sum()
    tn = w[(y_true == "no") & (y_pred == "no")].sum()
    fp = w[(y_true == "no") & (y_pred == "yes")].sum()
    fn = w[(y_true == "yes") & (y_pred == "no")].sum()
    n = tp + tn + fp + fn
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return {"acc": (tp + tn) / n, "precision": prec, "recall": rec, "f1": f1,
            "yes_ratio": (tp + fp) / n, "FP": round(fp), "FN": round(fn)}

# --- load (dòng sau ghi đè dòng trước cùng sample_key)
records = {}
with open(PRED_FILE, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            r = json.loads(line)
            records[r["sample_key"]] = r
df900 = pd.DataFrame(records.values())
print("Status:", df900["status"].value_counts().to_dict())
df900 = df900[df900["status"] == "success"].copy()
df900["label"] = df900["ground_truth"].str.lower().str.strip()
df900["pred"] = df900["prediction"].str.lower().str.strip()
df900["nq"] = df900["question"].apply(norm_q)
df900["object"] = df900["nq"].apply(extract_obj)
assert df900["object"].notna().all(), "Có câu hỏi không trích được object"
df900["pair_key"] = df900["image_source"].astype(str) + "||" + df900["nq"]

# --- cặp (ảnh, câu hỏi) duy nhất; positives của POPE trùng nhau giữa 3 split
g = df900.groupby("pair_key", sort=False)
assert (g["label"].nunique() == 1).all() and (g["pred"].nunique() == 1).all()
pairs = g.agg(
    sample_key=("sample_key", "first"), image_source=("image_source", "first"),
    question=("question", "first"), object=("object", "first"),
    label=("label", "first"), pred=("pred", "first"), raw_output=("raw_output", "first"),
    n_occ=("sample_key", "size"),
    categories=("category", lambda x: ",".join(sorted(set(x), key=CATEGORIES.index))),
).reset_index()
pairs["error_type"] = np.select(
    [(pairs.label == "no") & (pairs.pred == "yes"), (pairs.label == "yes") & (pairs.pred == "no")],
    ["FP", "FN"], default="OK")
pairs["is_error"] = pairs["error_type"] != "OK"

print("\n900 hàng      :", len(df900), "| cặp duy nhất:", len(pairs))
print("Baseline 900 hàng :", {k: round(v, 4) for k, v in metrics_w(df900.label, df900.pred).items()})
print("Baseline duy nhất :", {k: round(v, 4) for k, v in metrics_w(pairs.label, pairs.pred).items()})

# --- So sánh theo category: chỉ có nghĩa với NEGATIVE (positive giống nhau ở cả 3 split)
neg = df900[df900.label == "no"]
cat_fp = (neg.assign(fp=(neg.pred == "yes"))
             .groupby("category")["fp"].agg(["sum", "count", "mean"])
             .rename(columns={"sum": "FP", "count": "n_negative", "mean": "FP_rate"})
             .loc[CATEGORIES])
print("\nTỉ lệ FP (hallucination) theo loại negative:")
display(cat_fp.round(3))
print("FN rate (chỉ tính 1 lần vì positive giống nhau):",
      round(((pairs.label == "yes") & (pairs.pred == "no")).sum() / (pairs.label == "yes").sum(), 3))

In [ ]:
# ===== A2. LIST TỪNG CASE LỖI =====
errors = (pairs[pairs.is_error].sort_values(["error_type", "object"]).reset_index(drop=True))
errors["case_id"] = errors.groupby("error_type").cumcount().add(1).astype(str).str.zfill(2)
errors["case_id"] = errors["error_type"] + "_" + errors["case_id"]
pairs = pairs.merge(errors[["pair_key", "case_id"]], on="pair_key", how="left")

print(f"Tổng lỗi duy nhất: {len(errors)}  (FN={int((errors.error_type=='FN').sum())}, "
      f"FP={int((errors.error_type=='FP').sum())})\n")
for _, r in errors.iterrows():
    print(f"[{r.case_id}] obj={r.object:<14} GT={r.label:<3} Pred={r.pred:<3} "
          f"| splits={r.categories:<28} | {r.sample_key:<18} | {r.image_source} | raw='{r.raw_output}'")

cols = ["case_id", "error_type", "object", "question", "label", "pred", "raw_output", "categories", "image_source", "sample_key"]
errors[cols].to_csv(OUT2 / "error_list.csv", index=False, encoding="utf-8-sig")
print("\nSaved:", OUT2 / "error_list.csv")

### A3. Gắn đặc trưng cho từng case từ COCO annotations
Ảnh POPE lấy từ COCO val2014, nên tải `instances_val2014.json` để biết:
- vật thể được hỏi chiếm bao nhiêu % ảnh, có bao nhiêu instance (giải thích **FN**);
- ảnh còn chứa vật thể nào hay đi cùng vật thể được hỏi (giải thích **FP**: model bị "gợi ý" bởi co-occurrence);
- kiểm tra nhãn GT có khớp COCO không (loại trừ lỗi do nhãn).

In [ ]:
# ===== A3. COCO annotations + đặc trưng =====
COCO_DIR = Path("/kaggle/working/coco_ann"); COCO_DIR.mkdir(exist_ok=True)
INST = COCO_DIR / "annotations" / "instances_val2014.json"
if not INST.exists():
    zip_path = COCO_DIR / "annotations_trainval2014.zip"
    try:
        if not zip_path.exists():
            print("Đang tải COCO annotations (~240MB)...")
            urllib.request.urlretrieve("http://images.cocodataset.org/annotations/annotations_trainval2014.zip", zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extract("annotations/instances_val2014.json", COCO_DIR)
    except Exception as e:
        raise RuntimeError("Không tải được COCO annotations. Bật Internet trong Kaggle, "
                           "hoặc upload instances_val2014.json làm dataset rồi sửa INST. Lỗi gốc: " + repr(e))

coco = json.load(open(INST))
cat_name = {c["id"]: c["name"] for c in coco["categories"]}
img_wh = {im["id"]: (im["width"], im["height"]) for im in coco["images"]}
img_anns = defaultdict(lambda: defaultdict(list))
for a in coco["annotations"]:
    img_anns[a["image_id"]][cat_name[a["category_id"]]].append(
        {"bbox": a["bbox"], "area": a["area"], "crowd": a["iscrowd"]})
del coco

# xác suất co-occurrence trên toàn bộ val2014
cat_count, pair_count = Counter(), Counter()
for iid, d in img_anns.items():
    cs = list(d.keys())
    for c1 in cs:
        cat_count[c1] += 1
        for c2 in cs:
            if c1 != c2:
                pair_count[(c1, c2)] += 1

def p_obj_given_partner(obj, partner):
    return pair_count[(obj, partner)] / cat_count[partner] if cat_count[partner] else 0.0

def pair_features(r):
    iid = int(r.image_source.split("_")[-1])
    W, H = img_wh[iid]; area_img = W * H
    d = img_anns.get(iid, {})
    insts = d.get(r.object, [])
    best_p, best_partner = 0.0, None
    for partner in d:
        if partner == r.object:
            continue
        p = p_obj_given_partner(r.object, partner)
        if p > best_p:
            best_p, best_partner = p, partner
    return pd.Series({
        "coco_id": iid,
        "coco_has_obj": r.object in d,
        "n_inst": len(insts),
        "max_area_ratio": max((i["area"] for i in insts), default=0.0) / area_img,
        "sum_area_ratio": sum(i["area"] for i in insts) / area_img,
        "any_crowd": any(i["crowd"] for i in insts),
        "n_objects_img": sum(len(v) for v in d.values()),
        "n_cats_img": len(d),
        "cooc_max_p": best_p,
        "cooc_partner": best_partner,
    })

feat = pairs.apply(pair_features, axis=1)
pairs = pd.concat([pairs.drop(columns=[c for c in feat.columns if c in pairs.columns]), feat], axis=1)

# kiểm tra nhãn GT có khớp COCO không
mismatch = pairs[(pairs.label == "yes") != pairs.coco_has_obj]
print("Nhãn GT không khớp COCO:", len(mismatch), "/", len(pairs))
if len(mismatch):
    display(mismatch[["case_id", "sample_key", "object", "label", "pred", "coco_has_obj"]].head(20))
display(pairs[pairs.is_error][["case_id", "object", "n_inst", "max_area_ratio", "n_objects_img", "cooc_partner", "cooc_max_p"]]
        .round(3).head(10))

In [ ]:
# ===== A4. Gán lý do lỗi + đối chứng với nhóm trả lời đúng =====
from scipy.stats import mannwhitneyu

SMALL_T   = 0.01   # vật thể lớn nhất < 1% diện tích ảnh -> "nhỏ"
CLUTTER_N = 15     # ảnh có >= 15 instance annotate -> "đông đúc"
COOC_T    = 0.25   # P(obj | vật thể đang có trong ảnh) >= 0.25 -> "dễ gây ảo giác"

def auto_reason(r):
    if r.error_type == "FN":
        if r.max_area_ratio < SMALL_T:      return "FN-small_object"
        if r.n_objects_img >= CLUTTER_N:    return "FN-cluttered_scene"
        return "FN-visible_unexplained"
    if r.error_type == "FP":
        if r.cooc_max_p >= COOC_T:          return "FP-co_occurrence"
        return "FP-unexplained"
    return "OK"

pairs["auto_reason"] = pairs.apply(auto_reason, axis=1)
pairs["manual_reason"] = ""     # điền tay sau khi xem error_report.html

print("===== PHÂN LOẠI LỖI =====")
display(pairs[pairs.is_error]["auto_reason"].value_counts().to_frame("n_cases"))

# --- đối chứng: lý do có thật sự phân biệt lỗi với nhóm đúng không?
tp = pairs[(pairs.label == "yes") & (pairs.pred == "yes")]
fn = pairs[pairs.error_type == "FN"]
tn = pairs[(pairs.label == "no") & (pairs.pred == "no")]
fp = pairs[pairs.error_type == "FP"]

def cmp(a, b, col, name_a, name_b):
    if len(a) < 3 or len(b) < 3:
        print(f"{col}: không đủ mẫu"); return
    p = mannwhitneyu(a[col], b[col], alternative="two-sided").pvalue
    print(f"{col:<15} median {name_a}={a[col].median():.4f} vs {name_b}={b[col].median():.4f} | Mann-Whitney p={p:.4f}")

print("\n===== FN so với TP (vật thể có trong ảnh) =====")
cmp(fn, tp, "max_area_ratio", "FN", "TP")
cmp(fn, tp, "n_objects_img", "FN", "TP")
print("Tỉ lệ vật thể nhỏ (<%.0f%% ảnh): FN=%.2f | TP=%.2f" %
      (SMALL_T * 100, (fn.max_area_ratio < SMALL_T).mean(), (tp.max_area_ratio < SMALL_T).mean()))

print("\n===== FP so với TN (vật thể không có trong ảnh) =====")
cmp(fp, tn, "cooc_max_p", "FP", "TN")
print("Tỉ lệ co-occurrence cao (>=%.2f): FP=%.2f | TN=%.2f" %
      (COOC_T, (fp.cooc_max_p >= COOC_T).mean(), (tn.cooc_max_p >= COOC_T).mean()))

print("\n===== LỖI THEO OBJECT (tỉ lệ trên số câu hỏi của object đó) =====")
obj_tab = (pairs.groupby("object")
    .agg(total=("pair_key", "size"),
         FN=("error_type", lambda x: (x == "FN").sum()),
         FP=("error_type", lambda x: (x == "FP").sum()),
         n_gt_yes=("label", lambda x: (x == "yes").sum()),
         n_gt_no=("label", lambda x: (x == "no").sum())))
obj_tab["errors"] = obj_tab.FN + obj_tab.FP
obj_tab["FN_rate"] = (obj_tab.FN / obj_tab.n_gt_yes.replace(0, np.nan)).round(2)
obj_tab["FP_rate"] = (obj_tab.FP / obj_tab.n_gt_no.replace(0, np.nan)).round(2)
display(obj_tab.sort_values(["errors", "total"], ascending=False).head(15))

obj_tab.to_csv(OUT2 / "object_error_table.csv", encoding="utf-8-sig")
pairs.to_csv(OUT2 / "pairs_with_features.csv", index=False, encoding="utf-8-sig")
pairs[pairs.is_error].to_csv(OUT2 / "error_list_with_reason.csv", index=False, encoding="utf-8-sig")

In [ ]:
# ===== A5. Báo cáo HTML: ảnh + bbox + lý do cho TỪNG lỗi =====
# Lấy lại ảnh gốc từ dataset đã chọn (cùng seed như Bước 1)
def ensure_selected():
    global selected_dataset
    if "selected_dataset" in globals():
        return selected_dataset
    from datasets import load_dataset
    ds = load_dataset(DATASET_ID, name="Full", revision=DATASET_REVISION)
    selected_dataset = {c: ds[c].shuffle(seed=RANDOM_SEED).select(range(SAMPLES_PER_CATEGORY)) for c in CATEGORIES}
    return selected_dataset

selected_dataset = ensure_selected()
sk2loc = {}
for c in CATEGORIES:
    for i, qid in enumerate(selected_dataset[c]["question_id"]):
        sk2loc[f"{c}:{qid}"] = (c, i)

def get_pil(sample_key):
    c, i = sk2loc[sample_key]
    img = selected_dataset[c][i]["image"]
    img.load()
    return img.convert("RGB")

def annotated_b64(r, maxw=520):
    img = get_pil(r.sample_key)
    d = ImageDraw.Draw(img)
    anns = img_anns.get(int(r.coco_id), {})
    target, color = (r.object, (255, 0, 0)) if r.error_type == "FN" else (r.cooc_partner, (0, 90, 255))
    if isinstance(target, str):
        for a in anns.get(target, []):
            x, y, w, h = a["bbox"]
            d.rectangle([x, y, x + w, y + h], outline=color, width=4)
    img.thumbnail((maxw, maxw))
    buf = io.BytesIO(); img.save(buf, "JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode()

def esc(s):
    return str(s).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

parts = ["<html><head><meta charset='utf-8'><title>Phi-4 POPE error report</title>"
         "<style>body{font-family:sans-serif;background:#fafafa;margin:20px}"
         ".card{display:inline-block;vertical-align:top;width:540px;margin:8px;padding:10px;"
         "background:#fff;border:1px solid #ddd;border-radius:8px}"
         ".FN{border-top:5px solid #d33}.FP{border-top:5px solid #05f}"
         "small{color:#555}</style></head><body>"
         "<h2>Danh sách lỗi Phi-4 trên POPE</h2>"
         "<p>Khung <b style='color:#d33'>đỏ</b> = vị trí vật thể thật sự có trong ảnh (FN). "
         "Khung <b style='color:#05f'>xanh</b> = vật thể hay đi kèm có trong ảnh, có thể gây ảo giác (FP).</p>"]
for _, r in pairs[pairs.is_error].sort_values(["error_type", "object"]).iterrows():
    if r.error_type == "FN":
        facts = (f"Vật thể chiếm {r.max_area_ratio*100:.2f}% ảnh, {int(r.n_inst)} instance; "
                 f"ảnh có {int(r.n_objects_img)} vật thể annotate")
    else:
        facts = (f"Vật thể không có trong COCO; ảnh có <b>{esc(r.cooc_partner)}</b> "
                 f"(P({esc(r.object)}|{esc(r.cooc_partner)})={r.cooc_max_p:.2f})")
    parts.append(
        f"<div class='card {r.error_type}'><b>{r.case_id}</b> | {esc(r.object)} | "
        f"GT=<b>{r.label}</b> Pred=<b>{r.pred}</b> <small>({esc(r.categories)})</small><br>"
        f"<img src='data:image/jpeg;base64,{annotated_b64(r)}' width='520'><br>"
        f"<small>{esc(r.question)}</small><br>"
        f"<small>Lý do tự động: <b>{r.auto_reason}</b><br>{facts}</small></div>")
parts.append("</body></html>")
(OUT2 / "error_report.html").write_text("\n".join(parts), encoding="utf-8")
print("Saved:", OUT2 / "error_report.html", "(tải về mở bằng trình duyệt để xem từng case)")

### Kết luận Bước 2 (điền sau khi chạy)
Điền vài dòng từ các bảng ở A1/A4, ví dụ:
- Tỉ lệ FN so với FP, model thiên về trả lời *No* hay *Yes*.
- FP tăng từ random → popular → adversarial hay không (bảng `cat_fp`).
- Object nào lỗi nhiều nhất.
- Lý do chiếm đa số và kết quả đối chứng (p-value). Nếu p lớn thì giả thuyết đó **chưa được ủng hộ**, không nên khẳng định.

## Phần B. Bước 3: Làm sao **phát hiện** các lỗi đó (không dùng nhãn GT)?

Ba kỹ thuật phát hiện lỗi mà không cần biết đáp án, mỗi kỹ thuật nhắm vào một loại lỗi:

| Kỹ thuật | Ý tưởng | Dự kiến bắt tốt |
|---|---|---|
| **B-conf** Độ tự tin | Chênh lệch log-prob giữa token `Yes` và `No` ở bước sinh đầu tiên. Margin gần 0 = model lưỡng lự. | lỗi "sát ngưỡng" |
| **B-cons** Nhất quán | Hỏi lại bằng 3 cách diễn đạt prompt. Câu trả lời đổi = đáng ngờ. | lỗi nhạy với prompt |
| **B-det** Đối chiếu detector | Chạy DETR (detector độc lập). Phi nói *No* nhưng DETR thấy vật thể = nghi FN. Phi nói *Yes* nhưng DETR không thấy = nghi FP. | FN, FP có vật thể rõ |

Đánh giá từng kỹ thuật **như một bộ phát hiện lỗi**: AUROC (phân biệt đáp án sai/đúng) với khoảng tin cậy bootstrap, precision/recall ở ngưỡng tự nhiên, và tỉ lệ bắt được theo từng lý do lỗi ở Phần A.

**Lưu ý trung thực**
- DETR huấn luyện trên COCO train2017, gồm phần lớn ảnh val2014, nên ảnh POPE có thể đã nằm trong dữ liệu huấn luyện DETR. Bằng chứng từ DETR vì vậy có thể *lạc quan*; đây là kết quả thăm dò, không phải số công bố.
- Chỉ có khoảng 49 lỗi, nên AUROC dao động mạnh; hãy nhìn khoảng tin cậy.

In [ ]:
# ===== B1. Margin Yes/No + 3 cách hỏi (có checkpoint) =====
import torch
tok = processor.tokenizer

def single_ids(words):
    ids = set()
    for w in words:
        t = tok.encode(w, add_special_tokens=False)
        if len(t) == 1:
            ids.add(t[0])
    return sorted(ids)

YES_IDS = single_ids(["Yes", "yes", " Yes", " yes", "YES"])
NO_IDS  = single_ids(["No", "no", " No", " no", "NO"])
print("YES ids:", YES_IDS, "| NO ids:", NO_IDS)
assert YES_IDS and NO_IDS

PROMPTS = {
    "P0": "{q} Answer with only Yes or No.",   # đúng prompt Bước 1
    "P1": "Look carefully at the entire image, including small, distant and partially hidden objects. {q} Answer with only Yes or No.",
    "P2": "Question about this photo: {q} Reply with a single word: Yes or No.",
}

@torch.inference_mode()
def phi4_margin(image, question, variant="P0"):
    text = PROMPTS[variant].format(q=question)
    prompt = f"<|user|><|image_1|>{text}<|end|><|assistant|>"
    inputs = processor(text=prompt, images=[image.convert("RGB")], return_tensors="pt").to("cuda:0")
    out = model.generate(**inputs, max_new_tokens=1, do_sample=False, use_cache=True,
                         output_scores=True, return_dict_in_generate=True)
    logp = torch.log_softmax(out.scores[0][0].float(), dim=-1)
    ly = torch.logsumexp(logp[YES_IDS], 0)
    ln = torch.logsumexp(logp[NO_IDS], 0)
    mass = torch.exp(torch.logsumexp(logp[YES_IDS + NO_IDS], 0))
    return float(ly - ln), float(mass)

MARGIN_FILE = OUT3 / "margins.jsonl"
done = {}
if MARGIN_FILE.exists():
    for line in open(MARGIN_FILE, encoding="utf-8"):
        if line.strip():
            r = json.loads(line); done[(r["pair_key"], r["variant"])] = r

t0 = time.time()
with open(MARGIN_FILE, "a", encoding="utf-8") as fout:
    for i, r in enumerate(pairs.itertuples()):
        img = None
        for v in PROMPTS:
            if (r.pair_key, v) in done:
                continue
            if img is None:
                img = get_pil(r.sample_key)
            try:
                m, mass = phi4_margin(img, r.question, v)
                rec = {"pair_key": r.pair_key, "variant": v, "margin": m, "mass": mass}
            except Exception as e:
                if "out of memory" in str(e).lower():
                    torch.cuda.empty_cache()
                rec = {"pair_key": r.pair_key, "variant": v, "margin": None, "mass": None, "error": str(e)[:200]}
            done[(r.pair_key, v)] = rec
            fout.write(json.dumps(rec) + "\n"); fout.flush()
        if (i + 1) % 50 == 0:
            print(f"{i+1}/{len(pairs)} pairs | {(time.time()-t0)/60:.1f} min")

for v in PROMPTS:
    pairs["m_" + v] = [done[(k, v)]["margin"] for k in pairs.pair_key]
    pairs["mass_" + v] = [done[(k, v)]["mass"] for k in pairs.pair_key]
assert pairs[["m_P0", "m_P1", "m_P2"]].notna().all().all(), "Còn mẫu lỗi, chạy lại cell để resume"

# kiểm tra tái lập Bước 1
agree = ((pairs.m_P0 > 0) == (pairs.pred == "yes")).mean()
print(f"\nP0 (margin>0) trùng prediction Bước 1: {agree:.3f}")
print("Khối lượng xác suất Yes/No trung bình (nên ~1):", pairs[["mass_P0", "mass_P1", "mass_P2"]].mean().round(3).to_dict())

In [ ]:
# ===== B2. Detector độc lập (DETR) =====
from transformers import DetrImageProcessor, DetrForObjectDetection

DET_ID = "facebook/detr-resnet-50"
det_dev = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
det_proc  = DetrImageProcessor.from_pretrained(DET_ID, revision="no_timm")   # revision no_timm: không cần cài timm
det_model = DetrForObjectDetection.from_pretrained(DET_ID, revision="no_timm").to(det_dev).eval()
label2id = det_model.config.label2id
missing_obj = sorted(set(pairs.object) - set(label2id))
print("Object không có trong nhãn DETR:", missing_obj)

@torch.inference_mode()
def detr_class_scores(image):
    inp = det_proc(images=image.convert("RGB"), return_tensors="pt").to(det_dev)
    out = det_model(**inp)
    probs = out.logits.softmax(-1)[0, :, :-1]       # bỏ lớp "no-object"
    return probs.max(0).values.cpu().numpy()         # điểm cao nhất cho mỗi lớp

DET_FILE = OUT3 / "detr_scores.json"
det_cache = json.load(open(DET_FILE)) if DET_FILE.exists() else {}
for r in pairs.drop_duplicates("image_source").itertuples():
    if r.image_source not in det_cache:
        det_cache[r.image_source] = detr_class_scores(get_pil(r.sample_key)).tolist()
json.dump(det_cache, open(DET_FILE, "w"))

pairs["det_score"] = [
    det_cache[r.image_source][label2id[r.object]] if r.object in label2id else np.nan
    for r in pairs.itertuples()]
print("Số cặp có điểm DETR:", pairs.det_score.notna().sum(), "/", len(pairs))
display(pairs.groupby("label")["det_score"].describe().round(3))

In [ ]:
# ===== B3. Đánh giá các kỹ thuật phát hiện lỗi =====
from sklearn.metrics import roc_auc_score

def auc_ci(y, s, n=1000, seed=0):
    y = np.asarray(y).astype(int); s = np.asarray(s, float)
    if y.sum() == 0 or y.sum() == len(y):
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    base = roc_auc_score(y, s); bs = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() != y[i].max():
            bs.append(roc_auc_score(y[i], s[i]))
    lo, hi = np.percentile(bs, [2.5, 97.5])
    return base, lo, hi

pairs["pred0"] = np.where(pairs.m_P0 > 0, "yes", "no")          # dùng cùng nguồn với margin
pairs["wrong"] = pairs.pred0 != pairs.label

# điểm "đáng ngờ": càng cao càng có khả năng sai
pairs["s_conf"] = -pairs.m_P0.abs()
pairs["s_cons"] = ((np.where(pairs.m_P1 > 0, "yes", "no") != pairs.pred0).astype(int)
                   + (np.where(pairs.m_P2 > 0, "yes", "no") != pairs.pred0).astype(int))
pairs["s_det"]  = np.where(pairs.pred0 == "no", pairs.det_score, 1 - pairs.det_score)
DETECTORS = {"B-conf (độ tự tin)": "s_conf", "B-cons (nhất quán prompt)": "s_cons", "B-det (đối chiếu DETR)": "s_det"}

rows = []
subsets = {
    "Phi trả lời NO  -> bắt FN": pairs[pairs.pred0 == "no"],
    "Phi trả lời YES -> bắt FP": pairs[pairs.pred0 == "yes"],
    "Tất cả": pairs,
}
for sname, sub in subsets.items():
    sub = sub[sub.det_score.notna()]
    for dname, col in DETECTORS.items():
        a, lo, hi = auc_ci(sub.wrong, sub[col])
        rows.append({"tập": sname, "kỹ thuật": dname, "n": len(sub), "n_sai": int(sub.wrong.sum()),
                     "AUROC": round(a, 3), "CI95_lo": round(lo, 3), "CI95_hi": round(hi, 3)})
auc_df = pd.DataFrame(rows)
print("===== AUROC: kỹ thuật nào tách được câu trả lời sai khỏi đúng? (0.5 = ngẫu nhiên) =====")
display(auc_df)

# --- ngưỡng tự nhiên (không tinh chỉnh trên nhãn)
CONF_T, DET_NO_T, DET_YES_T = 1.0, 0.7, 0.3
pairs["flag_conf"] = pairs.m_P0.abs() < CONF_T
pairs["flag_cons"] = pairs.s_cons >= 1
pairs["flag_det"]  = ((pairs.pred0 == "no") & (pairs.det_score >= DET_NO_T)) | \
                     ((pairs.pred0 == "yes") & (pairs.det_score < DET_YES_T))
flag_cols = {"B-conf": "flag_conf", "B-cons": "flag_cons", "B-det": "flag_det"}

print(f"\n===== Precision/Recall ở ngưỡng tự nhiên (|margin|<{CONF_T}; DETR>={DET_NO_T} nếu Phi=No, <{DET_YES_T} nếu Phi=Yes) =====")
pr = []
for n, c in flag_cols.items():
    tp_ = int((pairs[c] & pairs.wrong).sum()); fl = int(pairs[c].sum()); er = int(pairs.wrong.sum())
    pr.append({"kỹ thuật": n, "số cặp bị gắn cờ": fl, "gắn cờ đúng lỗi": tp_,
               "precision": round(tp_ / fl, 3) if fl else np.nan, "recall": round(tp_ / er, 3) if er else np.nan})
display(pd.DataFrame(pr))

# --- kỹ thuật nào bắt được loại lỗi nào (nối với lý do ở Phần A)
err = pairs[pairs.is_error].copy()
catch = (err.groupby("auto_reason")[list(flag_cols.values())].mean().round(2)
            .rename(columns={v: k for k, v in flag_cols.items()}))
catch["n_cases"] = err.groupby("auto_reason").size()
catch["bắt được >=1 kỹ thuật"] = err.groupby("auto_reason").apply(lambda d: d[list(flag_cols.values())].any(axis=1).mean()).round(2)
print("\n===== Tỉ lệ lỗi được bắt, theo lý do lỗi =====")
display(catch)

auc_df.to_csv(OUT3 / "detector_auroc.csv", index=False, encoding="utf-8-sig")
catch.to_csv(OUT3 / "detector_catch_by_reason.csv", encoding="utf-8-sig")
pairs.to_csv(OUT3 / "pairs_with_detection_scores.csv", index=False, encoding="utf-8-sig")

### Kết luận Bước 3 (điền sau khi chạy)
- Kỹ thuật nào có AUROC cao nhất, và khoảng tin cậy có tách khỏi 0.5 không?
- Mỗi kỹ thuật bắt tốt loại lỗi nào (bảng "bắt được theo lý do lỗi")? Loại lỗi nào **không** kỹ thuật nào bắt được (ví dụ `FN-visible_unexplained`)? Đó là phần cần nghiên cứu thêm hoặc cần kiểm tra nhãn.
- Nhắc lại caveat về DETR đã thấy ảnh COCO.

## Phần C. Bước 4: Từ các lỗi thấy được, cải thiện model

Mỗi cách cải thiện nhắm vào một nguyên nhân lỗi đã tìm thấy:

| Cách | Nhắm vào | Cần nhãn? |
|---|---|---|
| **S1** Hiệu chỉnh ngưỡng Yes/No | Model lệch về *No* (FN >> FP) | có (chọn 1 tham số, cross-validation) |
| **S2** Prompt "nhìn kỹ" (P1) và ensemble 3 prompt | FN do vật thể nhỏ/nền | không/có |
| **S3** Kết hợp margin Phi + bằng chứng DETR (logistic regression) | FN và FP có bằng chứng độc lập | có (CV) |
| **S4** Phát hiện rồi phóng to: chỉ với cặp nghi FN (Phi=No, DETR thấy vật thể) thì cắt 4 góc, phóng to ×2, hỏi lại | FN vật thể nhỏ | **không** |

**Cách đánh giá cho công bằng**
- Mọi tham số học từ nhãn dùng **GroupKFold theo ảnh** (5 fold), nên cùng một ảnh không nằm ở cả train lẫn test.
- Báo cả hai góc nhìn: trên 900 hàng (so sánh trực tiếp với baseline Bước 1) và trên cặp duy nhất.
- Báo số lỗi *sửa được* và số lỗi *gây ra thêm* so với baseline.
- Hướng dài hạn như fine-tune LoRA với hard-negative hay Visual Contrastive Decoding được thảo luận ở cuối, không chạy ở đây vì T4 hạn chế.

In [ ]:
# ===== C1. S0-S3 với cross-validation theo ảnh =====
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

assert pairs.det_score.notna().all(), "Có object không có trong nhãn DETR; xử lý trước khi chạy S3"
y_yes = (pairs.label == "yes").values
w = pairs.n_occ.values.astype(float)           # trọng số = số lần cặp xuất hiện trong 900 hàng
folds = list(GroupKFold(n_splits=5).split(pairs, groups=pairs.image_source))

def fit_bias(m, y, w):
    grid = np.linspace(-6, 6, 241)
    accs = np.array([(((m + b) > 0) == y)[:] @ w / w.sum() for b in grid])
    best = np.flatnonzero(accs >= accs.max() - 1e-12)
    return grid[best[np.argmin(np.abs(grid[best]))]]           # nếu hoà, chọn b gần 0 nhất

def cv_bias(m):
    pred = np.empty(len(pairs), dtype=object)
    for tr, te in folds:
        b = fit_bias(m[tr], y_yes[tr], w[tr])
        pred[te] = np.where(m[te] + b > 0, "yes", "no")
    return pred

M0, M1, M2 = pairs.m_P0.values, pairs.m_P1.values, pairs.m_P2.values
Mens = (M0 + M1 + M2) / 3
det_logit = np.log(np.clip(pairs.det_score.values, 1e-3, 1 - 1e-3) / (1 - np.clip(pairs.det_score.values, 1e-3, 1 - 1e-3)))

systems = {}
systems["S0 baseline (P0)"] = pairs.pred.values
systems["S1 P0 + hiệu chỉnh bias"] = cv_bias(M0)
systems["S2a prompt 'nhìn kỹ' (P1)"] = np.where(M1 > 0, "yes", "no")
systems["S2b ensemble 3 prompt + bias"] = cv_bias(Mens)

X = np.column_stack([M0, M1, M2, det_logit])
pred_lr = np.empty(len(pairs), dtype=object)
for tr, te in folds:
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000))
    clf.fit(X[tr], y_yes[tr], logisticregression__sample_weight=w[tr])
    pred_lr[te] = np.where(clf.predict(X[te]), "yes", "no")
systems["S3 Phi + DETR (logreg)"] = pred_lr

print("Bias hiệu chỉnh khi fit trên toàn bộ dữ liệu (chỉ để tham khảo): b =", round(fit_bias(M0, y_yes, w), 2))

In [ ]:
# ===== C2. S4: phát hiện rồi phóng to (không dùng nhãn) =====
GATE_DET = 0.3     # DETR thấy vật thể với điểm >= 0.3 mà Phi nói No -> nghi FN
TILE_FRAC = 0.6
UPSCALE = 2

def make_tiles(img, frac=TILE_FRAC, up=UPSCALE):
    W, H = img.size; tw, th = int(W * frac), int(H * frac)
    boxes = [(0, 0, tw, th), (W - tw, 0, W, th), (0, H - th, tw, H), (W - tw, H - th, W, H)]
    return [img.crop(b).resize(((b[2] - b[0]) * up, (b[3] - b[1]) * up), Image.BICUBIC) for b in boxes]

gate = (pairs.pred0 == "no") & (pairs.det_score >= GATE_DET)
print(f"Số cặp nghi FN được phóng to kiểm tra: {int(gate.sum())} / {len(pairs)}")

TILE_FILE = OUT4 / "tile_margins.json"
tile_cache = json.load(open(TILE_FILE)) if TILE_FILE.exists() else {}
for r in pairs[gate].itertuples():
    if r.pair_key in tile_cache:
        continue
    tiles = make_tiles(get_pil(r.sample_key))
    tile_cache[r.pair_key] = [phi4_margin(t, r.question, "P0")[0] for t in tiles]
    json.dump(tile_cache, open(TILE_FILE, "w"))

pairs["tile_max"] = [max(tile_cache[k]) if k in tile_cache else np.nan for k in pairs.pair_key]

def zoom_pred(tile_t):
    p = pairs.pred0.values.copy()
    flip = gate.values & (pairs.tile_max.values > tile_t)
    p[flip] = "yes"
    return p

TILE_T = 0.0       # cố định trước: một tile nói Yes (margin>0) thì lật sang Yes
systems["S4 phát hiện + phóng to"] = zoom_pred(TILE_T)

print("\nĐộ nhạy theo ngưỡng tile (chỉ để xem xu hướng, không dùng chọn ngưỡng trên cùng dữ liệu):")
sens = []
for t in [-3, -2, -1, 0, 1, 2]:
    m = metrics_w(pairs.label, zoom_pred(t), w)
    sens.append({"TILE_T": t, "acc": round(m["acc"], 4), "FP": m["FP"], "FN": m["FN"]})
display(pd.DataFrame(sens))

In [ ]:
# ===== C3. So sánh hệ thống, lỗi sửa được / gây thêm, lỗi còn lại =====
base_pred = systems["S0 baseline (P0)"]
rows = []
for name, pred in systems.items():
    m900 = None
    # chuyển dự đoán cặp -> 900 hàng để so sánh trực tiếp với Bước 1
    mapping = dict(zip(pairs.pair_key, pred))
    p900 = df900["pair_key"].map(mapping)
    m900 = metrics_w(df900.label, p900)
    mu = metrics_w(pairs.label, pred)
    fixed  = int(((base_pred != pairs.label) & (pred == pairs.label)).sum())
    broken = int(((base_pred == pairs.label) & (pred != pairs.label)).sum())
    per_cat = {c: round(metrics_w(df900[df900.category == c].label, p900[df900.category == c])["acc"], 3) for c in CATEGORIES}
    rows.append({"hệ thống": name,
                 "acc (900 hàng)": round(m900["acc"], 4), "F1": round(m900["f1"], 4),
                 "recall": round(m900["recall"], 4), "precision": round(m900["precision"], 4),
                 "yes_ratio": round(m900["yes_ratio"], 3), "FP": m900["FP"], "FN": m900["FN"],
                 "acc (duy nhất)": round(mu["acc"], 4),
                 "sửa được": fixed, "gây thêm": broken,
                 **{f"acc_{c}": v for c, v in per_cat.items()}})
cmp_df = pd.DataFrame(rows)
print("===== SO SÁNH CÁC CÁCH CẢI THIỆN (cùng 900 mẫu) =====")
display(cmp_df)
cmp_df.to_csv(OUT4 / "comparison.csv", index=False, encoding="utf-8-sig")

# --- chọn hệ thống tốt nhất theo acc (900 hàng) để xem chi tiết
best_name = cmp_df.sort_values("acc (900 hàng)", ascending=False).iloc[0]["hệ thống"]
print("\nHệ thống có acc cao nhất:", best_name)
bp = systems[best_name]
pairs["best_pred"] = bp
detail = pairs[(base_pred != pairs.label) | (bp != pairs.label)].copy()
detail["status"] = np.select(
    [(base_pred != pairs.label)[detail.index] & (bp == pairs.label)[detail.index],
     (base_pred == pairs.label)[detail.index] & (bp != pairs.label)[detail.index]],
    ["FIXED", "BROKEN"], default="STILL_WRONG")
print("\nTrạng thái từng case liên quan (so với baseline):")
display(detail["status"].value_counts().to_frame("n"))
display(detail[["case_id", "object", "label", "pred", "best_pred", "status", "auto_reason"]]
        .sort_values(["status", "object"]).reset_index(drop=True))
detail.to_csv(OUT4 / "fixed_broken_cases.csv", index=False, encoding="utf-8-sig")

print("\nLỗi còn lại theo lý do (hệ thống tốt nhất):")
display(detail[detail.status == "STILL_WRONG"]["auto_reason"].value_counts().to_frame("n"))

In [ ]:
# ===== C4. Đóng gói kết quả để nộp / tải về =====
import shutil
shutil.make_archive("/kaggle/working/phi4_pope_analysis_outputs", "zip", BASE)
print("Đã tạo: /kaggle/working/phi4_pope_analysis_outputs.zip")
for d in (OUT2, OUT3, OUT4):
    print("\n", d.name)
    for p in sorted(d.iterdir()):
        print("  -", p.name)

### Kết luận Bước 4 và hướng tiếp theo (điền sau khi chạy)
- Cách nào tăng acc/F1 nhiều nhất? Có gây thêm lỗi mới không (cột "gây thêm")?
- Cách nào cải thiện *đúng loại lỗi* đã thấy ở Bước 2 (xem "lỗi còn lại theo lý do")?
- Cẩn trọng khi báo cáo: 555 cặp duy nhất là mẫu nhỏ, chênh lệch 1-2 điểm % nằm trong nhiễu. Nên chạy toàn bộ 9000 mẫu POPE trước khi kết luận chắc chắn.

**Hướng cải thiện sâu hơn (chưa chạy trong notebook này)**
1. **Fine-tune LoRA** (`peft` đã cài) với dữ liệu hard-negative: các câu hỏi adversarial/co-occurrence, và positive vật thể nhỏ lấy từ phần COCO train (không dùng ảnh POPE để tránh rò rỉ).
2. **Visual Contrastive Decoding / OPERA**: giảm ảo giác bằng cách đối chiếu logit giữa ảnh gốc và ảnh bị nhiễu.
3. **Tăng độ phân giải/nhiều crop** có điều kiện, chính là ý tưởng của S4 nhưng áp dụng ngay trong bộ xử lý ảnh.
4. **Pipeline phát hiện → sửa** dùng các tín hiệu ở Phần B làm bộ lọc để chỉ tốn thêm tính toán cho những cặp đáng ngờ.